In [41]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import StackingRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

In [42]:
df = pd.read_csv("database.csv")

In [43]:
num_fill_cols = [
    'Azimuthal Gap',
    'Horizontal Distance',
    'Root Mean Square',
    'Horizontal Error',
    'Depth Error',
    'Magnitude Error',
    'Magnitude Seismic Stations',
    'Depth Seismic Stations',
]

In [44]:
for c in num_fill_cols:
    df[c] = df[c].fillna(df[c].mean())

In [45]:
df['Magnitude Type'] = df['Magnitude Type'].fillna(df['Magnitude Type'].mode()[0])

In [46]:
dfCleaned = pd.get_dummies(
    df,
    columns=['Magnitude Type', 'Source', 'Type', 'Status',
             'Location Source', 'Magnitude Source'],
    drop_first=True
)

In [47]:
bool_cols = dfCleaned.select_dtypes(include=['bool']).columns
dfCleaned[bool_cols] = dfCleaned[bool_cols].astype(int)
dfCleaned = dfCleaned.drop(['ID', 'Date', 'Time'], axis=1)

In [48]:
scale_cols = ['Latitude', 'Longitude', 'Depth', 'Azimuthal Gap',
              'Horizontal Distance', 'Root Mean Square', 'Horizontal Error',
              'Depth Error']
scaler = StandardScaler()


In [49]:
x = dfCleaned.drop('Magnitude', axis=1)
y = dfCleaned['Magnitude']
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [50]:
dfCleaned[scale_cols] = scaler.fit_transform(dfCleaned[scale_cols])

**Random forest**

In [51]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(x_train, y_train)
y_pred = model.predict(x_test)

In [52]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
accuracy_rf = 100 - (mae / y_test.mean() * 100)
cv_rmse = np.sqrt(-cross_val_score(
    model, x, y, cv=5, scoring='neg_mean_squared_error'
)).mean()

In [53]:
print(f"Random Forest -> MAE: {mae:.6f}  RMSE: {rmse:.6f}  R²: {r2:.6f}  Accuracy: {accuracy_rf:.2f}%  CV_RMSE: {cv_rmse:.6f}")

Random Forest -> MAE: 0.269857  RMSE: 0.380107  R²: 0.216678  Accuracy: 95.42%  CV_RMSE: 0.400061


**SVM**

In [54]:
svm_model = SVR(kernel='rbf', C=1.0, epsilon=0.1, gamma='scale')
svm_model.fit(x_train, y_train)
y_pred_svm = svm_model.predict(x_test)

In [55]:
mae_svm = mean_absolute_error(y_test, y_pred_svm)
rmse_svm = np.sqrt(mean_squared_error(y_test, y_pred_svm))
r2_svm = r2_score(y_test, y_pred_svm)
accuracy_svm = 100 - (mae / y_test.mean() * 100)
cv_rmse_svm = np.sqrt(-cross_val_score(
    svm_model, x, y, cv=5, scoring='neg_mean_squared_error'
)).mean()

In [56]:
print(f"SVM -> MAE: {mae_svm:.6f}  RMSE: {rmse_svm:.6f}  R²: {r2_svm:.6f}  Accuracy: {accuracy_svm:.2f}%  CV_RMSE: {cv_rmse_svm:.6f}")

SVM -> MAE: 0.280251  RMSE: 0.421845  R²: 0.035206  Accuracy: 95.42%  CV_RMSE: 0.416958


**XG Boost model**

In [57]:
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)
xgb_model.fit(x_train, y_train)
y_pred_xgb = xgb_model.predict(x_test)

In [58]:
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mse_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)
accuracy_xgb = 100 - (mae_xgb / y_test.mean() * 100)
cv_rmse_xgb = np.sqrt(-cross_val_score(
    xgb_model, x, y, cv=5, scoring='neg_mean_squared_error'
).mean())

In [59]:
print(f"XGBoost -> MAE: {mae_xgb:.6f}  MSE: {mse_xgb:.6f}  RMSE: {rmse_xgb:.6f}  R²: {r2_xgb:.6f}  Accuracy: {accuracy_xgb:.2f}%  CV_RMSE: {cv_rmse_xgb:.6f}")

XGBoost -> MAE: 0.259053  MSE: 0.134576  RMSE: 0.366845  R²: 0.270383  Accuracy: 95.60%  CV_RMSE: 0.389194


**KNN**

In [60]:
knn_model = KNeighborsRegressor(
    n_neighbors=5
)
knn_model.fit(x_train, y_train)
y_pred_knn = knn_model.predict(x_test)

In [61]:
mae_knn = mean_absolute_error(y_test, y_pred_knn)
mse_knn = mean_squared_error(y_test, y_pred_knn)
rmse_knn = np.sqrt(mse_knn)
r2_knn = r2_score(y_test, y_pred_knn)
accuracy_knn = 100 - (mae_knn / y_test.mean() * 100)
cv_rmse_knn = np.sqrt(-cross_val_score(
    knn_model, x, y, cv=5, scoring='neg_mean_squared_error'
).mean())

In [62]:
print(f"KNN -> MAE: {mae_knn:.6f}  MSE: {mse_knn:.6f}  RMSE: {rmse_knn:.6f}  R²: {r2_knn:.6f}  Accuracy: {accuracy_knn:.2f}%  CV_RMSE: {cv_rmse_knn:.6f}")

KNN -> MAE: 0.306288  MSE: 0.182879  RMSE: 0.427643  R²: 0.008502  Accuracy: 94.80%  CV_RMSE: 0.430810


**Stacking Regressor by considering Random Forest + SVM + XGBoost as base models.**

In [63]:
stack_model = StackingRegressor(
    estimators=[
        ('rf', RandomForestRegressor(
            n_estimators=100,
            random_state=42
        )),
        ('svm', SVR(
            kernel='rbf',
            C=1.0,
            epsilon=0.1,
            gamma='scale'
        )),
        ('xgb', XGBRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=6,
            random_state=42
        ))
    ],
    final_estimator=RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )
)
stack_model.fit(x_train, y_train)
y_pred_stack = stack_model.predict(x_test)


In [64]:
mae_stack = mean_absolute_error(y_test, y_pred_stack)
mse_stack = mean_squared_error(y_test, y_pred_stack)
rmse_stack = np.sqrt(mse_stack)
r2_stack = r2_score(y_test, y_pred_stack)
accuracy_stack = 100 - (mae_stack / y_test.mean() * 100)
cv_rmse_stack = np.sqrt(-cross_val_score(
    stack_model, x, y, cv=5, scoring='neg_mean_squared_error'
).mean())

In [65]:
print(f"Stacking Regressor -> MAE: {mae_stack:.6f}  MSE: {mse_stack:.6f}  RMSE: {rmse_stack:.6f}  R²: {r2_stack:.6f}  Accuracy: {accuracy_stack:.2f}%  CV_RMSE: {cv_rmse_stack:.6f}")

Stacking Regressor -> MAE: 0.269916  MSE: 0.146136  RMSE: 0.382277  R²: 0.207707  Accuracy: 95.42%  CV_RMSE: 0.407736


**LTSM**

In [66]:
x_train_lstm = np.array(x_train).reshape(
    x_train.shape[0], 1, x_train.shape[1]
)

x_test_lstm = np.array(x_test).reshape(
    x_test.shape[0], 1, x_test.shape[1]
)

lstm_model = Sequential()

lstm_model.add(
    LSTM(
        50,
        activation='relu',
        input_shape=(1, x_train.shape[1])
    )
)

lstm_model.add(Dense(1))

lstm_model.compile(
    optimizer='adam',
    loss='mse'
)

lstm_model.fit(
    x_train_lstm,
    y_train,
    epochs=20,
    batch_size=32,
    verbose=0
)

y_pred_lstm = lstm_model.predict(
    x_test_lstm,
    verbose=0
).flatten()

In [ ]:
mae_lstm = mean_absolute_error(y_test, y_pred_lstm)
mse_lstm = mean_squared_error(y_test, y_pred_lstm)
rmse_lstm = np.sqrt(mse_lstm)
r2_lstm = r2_score(y_test, y_pred_lstm)
accuracy_lstm = 100 - (mae_lstm / y_test.mean() * 100)

In [69]:
print(f"LSTM -> MAE: {mae_lstm:.6f}  MSE: {mse_lstm:.6f}  RMSE: {rmse_lstm:.6f}  R²: {r2_lstm:.6f}  Accuracy: {accuracy_lstm:.2f}%")

LSTM -> MAE: 0.346189  MSE: 0.187742  RMSE: 0.433292  R²: -0.017863  Accuracy: 94.12%
